# Feature Engineering and Chronological Data Preparation

This notebook converts the raw LOBSTER message and order-book data into the
processed 500 ms datasets used by the downstream modeling experiments.

The pipeline constructs interpretable LOB-state, order-flow, and temporal
features, defines asset-specific forecasting targets, establishes the
chronological train/validation/test protocol, and saves the processed datasets.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/task_data")

In [2]:
def load_regular_session(asset, date, lob_columns=None):
    day_dir = DATA_DIR / asset / date

    message_path = day_dir / f"message_{date}.parquet"
    orderbook_path = day_dir / f"orderbook_{date}.parquet"

    msg = pd.read_parquet(message_path)

    if lob_columns is None:
        ob = pd.read_parquet(orderbook_path)
    else:
        ob = pd.read_parquet(orderbook_path, columns=lob_columns)

    # Convert timestamps to New York local time
    local_time = (
        pd.to_datetime(msg["time"], unit="ns", utc=True)
        .dt.tz_convert("America/New_York")
    )

    minutes = (
        local_time.dt.hour * 60
        + local_time.dt.minute
    )

    # Keep only regular trading session:
    # 09:30 <= time < 16:00 ET
    mask = (
        (minutes >= 9 * 60 + 30)
        & (minutes < 16 * 60)
    )

    # Apply the exact same mask to both synchronized files
    msg = msg.loc[mask].reset_index(drop=True)
    ob = ob.loc[mask].reset_index(drop=True)

    return msg, ob

## 500 ms Market Representation

Each asset-day is restricted to the regular trading session and represented on
a 500 ms calendar-time grid. Order-book variables describe the latest
observable LOB state, while message features aggregate order submissions,
deletions, executions, and volume within each interval.

This combines state and event information in a common real-time representation.

In [3]:
def build_500ms_data(asset, date):
    # --------------------------------------------------
    # 1. Load regular-session message + order-book data
    # --------------------------------------------------
    msg, ob = load_regular_session(asset, date)

    time_index = pd.DatetimeIndex(
        pd.to_datetime(
            msg["time"],
            unit="ns",
            utc=True
        ).dt.tz_convert("America/New_York")
    )

    # --------------------------------------------------
    # 2. Construct 500 ms LOB state
    # --------------------------------------------------
    ob = ob.copy()
    ob.index = time_index

    # Last known order-book state inside each 500 ms bin.
    # Forward-fill bins with no new LOB event.
    lob = (
        ob.resample("500ms")
        .last()
        .ffill()
    )

    # Convert encoded prices to dollars
    price_cols = []

    for level in range(1, 11):
        price_cols.extend([
            f"sell{level}",
            f"buy{level}"
        ])

    lob[price_cols] = lob[price_cols] / 1e4

    # --------------------------------------------------
    # 3. LOB features
    # --------------------------------------------------

    # Mid-price
    lob["mid_price"] = (
        lob["sell1"] + lob["buy1"]
    ) / 2

    # Bid-ask spread
    lob["spread"] = (
        lob["sell1"] - lob["buy1"]
    )

    # Level-1 imbalance
    lob["imbalance_l1"] = (
        lob["vbuy1"] - lob["vsell1"]
    ) / (
        lob["vbuy1"] + lob["vsell1"]
    )

    # Five-level depth
    bid_depth_5 = sum(
        lob[f"vbuy{i}"]
        for i in range(1, 6)
    )

    ask_depth_5 = sum(
        lob[f"vsell{i}"]
        for i in range(1, 6)
    )

    lob["bid_depth_5"] = bid_depth_5
    lob["ask_depth_5"] = ask_depth_5

    # Five-level imbalance
    lob["imbalance_l5"] = (
        bid_depth_5 - ask_depth_5
    ) / (
        bid_depth_5 + ask_depth_5
    )

    # Microprice
    lob["microprice"] = (
        lob["sell1"] * lob["vbuy1"]
        + lob["buy1"] * lob["vsell1"]
    ) / (
        lob["vbuy1"] + lob["vsell1"]
    )

    lob["microprice_deviation"] = (
        lob["microprice"]
        - lob["mid_price"]
    )

    # --------------------------------------------------
    # 4. Message-flow features
    # --------------------------------------------------
    msg = msg.copy()
    msg.index = time_index

    msg["is_add"] = (
        msg["event_type"] == "AddOrder"
    ).astype(int)

    msg["is_delete"] = (
        msg["event_type"] == "OrderDelete"
    ).astype(int)

    msg["is_execution"] = (
        msg["event_type"] == "OrderExecuted"
    ).astype(int)

    definitions = {
        "buy_add_volume": (
            "AddOrder",
            "Buy"
        ),
        "sell_add_volume": (
            "AddOrder",
            "Sell"
        ),
        "buy_delete_volume": (
            "OrderDelete",
            "Buy"
        ),
        "sell_delete_volume": (
            "OrderDelete",
            "Sell"
        ),
        "executed_buy_side_volume": (
            "OrderExecuted",
            "Buy"
        ),
        "executed_sell_side_volume": (
            "OrderExecuted",
            "Sell"
        ),
    }

    for col, (event, side) in definitions.items():
        msg[col] = np.where(
            (
                (msg["event_type"] == event)
                & (msg["direction"] == side)
            ),
            msg["size"],
            0
        )

    # Aggregate message flow inside each 500 ms bin
    flow = msg.resample("500ms").agg(
        event_count=(
            "event_type",
            "size"
        ),

        add_count=(
            "is_add",
            "sum"
        ),

        delete_count=(
            "is_delete",
            "sum"
        ),

        execution_count=(
            "is_execution",
            "sum"
        ),

        buy_add_volume=(
            "buy_add_volume",
            "sum"
        ),

        sell_add_volume=(
            "sell_add_volume",
            "sum"
        ),

        buy_delete_volume=(
            "buy_delete_volume",
            "sum"
        ),

        sell_delete_volume=(
            "sell_delete_volume",
            "sum"
        ),

        executed_buy_side_volume=(
            "executed_buy_side_volume",
            "sum"
        ),

        executed_sell_side_volume=(
            "executed_sell_side_volume",
            "sum"
        ),
    )

    # --------------------------------------------------
    # 5. Book-flow pressure
    # --------------------------------------------------
    total_flow = (
        flow["buy_add_volume"]
        + flow["sell_add_volume"]
        + flow["buy_delete_volume"]
        + flow["sell_delete_volume"]
    )

    flow["net_book_flow"] = (
        flow["buy_add_volume"]
        - flow["buy_delete_volume"]
        - flow["sell_add_volume"]
        + flow["sell_delete_volume"]
    )

    flow["normalized_book_flow"] = (
        flow["net_book_flow"]
        / total_flow.replace(0, np.nan)
    ).fillna(0)

    # --------------------------------------------------
    # 6. Execution imbalance
    # --------------------------------------------------
    total_executed_volume = (
        flow["executed_buy_side_volume"]
        + flow["executed_sell_side_volume"]
    )

    flow["execution_imbalance"] = (
        flow["executed_sell_side_volume"]
        - flow["executed_buy_side_volume"]
    ) / total_executed_volume.replace(
        0,
        np.nan
    )

    flow["execution_imbalance"] = (
        flow["execution_imbalance"]
        .fillna(0)
    )

    # --------------------------------------------------
    # 7. Combine LOB state + message flow
    # --------------------------------------------------
    data = lob.join(
        flow,
        how="inner"
    )

    return data

## Temporal Features

Backward-looking returns, rolling volatility, and recent market-activity
measures are added to capture short-term temporal dynamics. All such predictors
use only information available at or before the prediction timestamp.

In [4]:
def add_temporal_features(data):
    data = data.copy()

    # --------------------------------------------------
    # Past returns
    # These use only information available at or before t
    # --------------------------------------------------
    data["return_500ms"] = np.log(
        data["mid_price"]
        / data["mid_price"].shift(1)
    )

    data["return_1s"] = np.log(
        data["mid_price"]
        / data["mid_price"].shift(2)
    )

    data["return_2s"] = np.log(
        data["mid_price"]
        / data["mid_price"].shift(4)
    )

    data["return_5s"] = np.log(
        data["mid_price"]
        / data["mid_price"].shift(10)
    )

    data["return_10s"] = np.log(
        data["mid_price"]
        / data["mid_price"].shift(20)
    )

    # --------------------------------------------------
    # Rolling volatility
    # --------------------------------------------------
    r = data["return_500ms"]

    # 10 bins × 0.5 s = 5 s
    data["volatility_5s"] = (
        r.rolling(10).std()
    )

    # 60 bins × 0.5 s = 30 s
    data["volatility_30s"] = (
        r.rolling(60).std()
    )

    # --------------------------------------------------
    # Recent market activity
    # --------------------------------------------------
    data["avg_events_5s"] = (
        data["event_count"]
        .rolling(10)
        .mean()
    )

    data["avg_executions_5s"] = (
        data["execution_count"]
        .rolling(10)
        .mean()
    )

    return data

## Forecasting Targets

The regression target is the future log return of the LOB mid-price,

\[
r_{t,h} = \log\left(\frac{m_{t+h}}{m_t}\right),
\]

where \(m_t\) is the mid-price at the current timestamp and \(h\) is the
forecast horizon.

The mid-price represents the center of the best bid and ask and provides a
clean reference for short-horizon price movement without the mechanical
bid--ask bounce that can arise when transaction prices alternate between the
two sides of the book. Log returns express price changes in relative rather
than absolute units, making targets more comparable across assets with
different price levels.

TSLA and AMZN use 500 ms, 2 s, and 10 s horizons, while BRK.B uses 5 s,
10 s, and 30 s, based on the training-period horizon diagnostics.

Future observations are used only to construct the target variable and are
never included among the predictors. The mid-price is used as a forecasting
reference rather than an assumed executable price; trading performance is
evaluated separately using bid/ask execution prices and transaction costs.

In [5]:
def add_targets(data, asset):
    data = data.copy()

    # Asset-specific horizons based on activity
    # and zero-return diagnostics
    if asset in ["TSLA", "AMZN"]:
        horizons = {
            "500ms": 1,
            "2s": 4,
            "10s": 20,
        }

    else:  # BRK.B
        horizons = {
            "5s": 10,
            "10s": 20,
            "30s": 60,
        }

    for name, steps in horizons.items():
        data[f"target_{name}"] = np.log(
            data["mid_price"].shift(-steps)
            / data["mid_price"]
        )

    return data

## Build the Processed Asset-Day Datasets

The feature and target construction functions are now applied consistently to
every asset and trading day. Keeping each asset-day separate at this stage
preserves explicit chronological boundaries for the subsequent evaluation
protocol.

In [6]:
assets = [
    "TSLA",
    "AMZN",
    "BRK.B",
]

dates = [
    "2026-03-16",
    "2026-03-17",
    "2026-03-18",
    "2026-03-19",
    "2026-03-20",
]

daily_data = {}

for asset in assets:
    for date in dates:
        print(f"Building {asset} {date}...")

        df = build_500ms_data(
            asset,
            date
        )

        df = add_temporal_features(df)

        df = add_targets(
            df,
            asset
        )

        df["asset"] = asset
        df["date"] = date

        daily_data[
            (asset, date)
        ] = df

print("Finished.")

Building TSLA 2026-03-16...
Building TSLA 2026-03-17...
Building TSLA 2026-03-18...
Building TSLA 2026-03-19...
Building TSLA 2026-03-20...
Building AMZN 2026-03-16...
Building AMZN 2026-03-17...
Building AMZN 2026-03-18...
Building AMZN 2026-03-19...
Building AMZN 2026-03-20...
Building BRK.B 2026-03-16...
Building BRK.B 2026-03-17...
Building BRK.B 2026-03-18...
Building BRK.B 2026-03-19...
Building BRK.B 2026-03-20...
Finished.


In [7]:
for (asset, date), df in daily_data.items():
    print(
        asset,
        date,
        df.shape,
        df.index.min(),
        df.index.max()
    )

TSLA 2026-03-16 (46800, 75) 2026-03-16 09:30:00-04:00 2026-03-16 15:59:59.500000-04:00
TSLA 2026-03-17 (46800, 75) 2026-03-17 09:30:00-04:00 2026-03-17 15:59:59.500000-04:00
TSLA 2026-03-18 (46800, 75) 2026-03-18 09:30:00-04:00 2026-03-18 15:59:59.500000-04:00
TSLA 2026-03-19 (46800, 75) 2026-03-19 09:30:00-04:00 2026-03-19 15:59:59.500000-04:00
TSLA 2026-03-20 (46800, 75) 2026-03-20 09:30:00-04:00 2026-03-20 15:59:59.500000-04:00
AMZN 2026-03-16 (46800, 75) 2026-03-16 09:30:00-04:00 2026-03-16 15:59:59.500000-04:00
AMZN 2026-03-17 (46800, 75) 2026-03-17 09:30:00-04:00 2026-03-17 15:59:59.500000-04:00
AMZN 2026-03-18 (46800, 75) 2026-03-18 09:30:00-04:00 2026-03-18 15:59:59.500000-04:00
AMZN 2026-03-19 (46800, 75) 2026-03-19 09:30:00-04:00 2026-03-19 15:59:59.500000-04:00
AMZN 2026-03-20 (46800, 75) 2026-03-20 09:30:00-04:00 2026-03-20 15:59:59.500000-04:00
BRK.B 2026-03-16 (46800, 75) 2026-03-16 09:30:00-04:00 2026-03-16 15:59:59.500000-04:00
BRK.B 2026-03-17 (46800, 75) 2026-03-17 09

## Chronological Train / Validation / Test Split

The evaluation protocol uses a strict date-based split:

- **Training:** March 16–18
- **Validation:** March 19
- **Test:** March 20

No random splitting is used. Model selection and hyperparameter decisions are
made without using the held-out March 20 test day. Normalization is performed
later within the modeling pipelines using training data only.

Future-return targets are constructed independently within each trading day, so
target windows do not cross day boundaries or chronological split boundaries.
All temporal predictors are backward-looking and use only information available
at or before the prediction timestamp. Feature normalization is performed later
inside the modeling pipelines and is fitted using training data only before
being applied unchanged to validation and test observations.

In [8]:
train_dates = [
    "2026-03-16",
    "2026-03-17",
    "2026-03-18",
]

val_dates = [
    "2026-03-19",
]

test_dates = [
    "2026-03-20",
]


def combine_dates(asset, selected_dates):
    return pd.concat(
        [
            daily_data[(asset, date)]
            for date in selected_dates
        ],
        axis=0
    )


splits = {}

for asset in assets:
    splits[asset] = {
        "train": combine_dates(asset, train_dates),
        "val": combine_dates(asset, val_dates),
        "test": combine_dates(asset, test_dates),
    }

    print(
        asset,
        "Train:", splits[asset]["train"].shape,
        "Validation:", splits[asset]["val"].shape,
        "Test:", splits[asset]["test"].shape,
    )

TSLA Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)
AMZN Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)
BRK.B Train: (140400, 75) Validation: (46800, 75) Test: (46800, 75)


In [9]:
for asset in assets:
    print(f"\n{asset}")

    for split_name in ["train", "val", "test"]:
        df = splits[asset][split_name]

        print(
            split_name,
            "rows =", len(df),
            "| start =", df.index.min(),
            "| end =", df.index.max()
        )


TSLA
train rows = 140400 | start = 2026-03-16 09:30:00-04:00 | end = 2026-03-18 15:59:59.500000-04:00
val rows = 46800 | start = 2026-03-19 09:30:00-04:00 | end = 2026-03-19 15:59:59.500000-04:00
test rows = 46800 | start = 2026-03-20 09:30:00-04:00 | end = 2026-03-20 15:59:59.500000-04:00

AMZN
train rows = 140400 | start = 2026-03-16 09:30:00-04:00 | end = 2026-03-18 15:59:59.500000-04:00
val rows = 46800 | start = 2026-03-19 09:30:00-04:00 | end = 2026-03-19 15:59:59.500000-04:00
test rows = 46800 | start = 2026-03-20 09:30:00-04:00 | end = 2026-03-20 15:59:59.500000-04:00

BRK.B
train rows = 140400 | start = 2026-03-16 09:30:00-04:00 | end = 2026-03-18 15:59:59.500000-04:00
val rows = 46800 | start = 2026-03-19 09:30:00-04:00 | end = 2026-03-19 15:59:59.500000-04:00
test rows = 46800 | start = 2026-03-20 09:30:00-04:00 | end = 2026-03-20 15:59:59.500000-04:00


## Candidate Feature Set

The candidate predictors combine three complementary information sources:

- **LOB state:** spread, depth, imbalance, and microprice deviation;
- **Order flow:** message activity, executions, additions/deletions, and signed flow;
- **Temporal context:** recent returns, volatility, and recent activity.

This provides interpretable predictors from both the order book and message
stream.

In [10]:
feature_cols = [
    # LOB state
    "spread",
    "imbalance_l1",
    "imbalance_l5",
    "bid_depth_5",
    "ask_depth_5",
    "microprice_deviation",

    # Order / message flow
    "event_count",
    "add_count",
    "delete_count",
    "execution_count",
    "normalized_book_flow",
    "execution_imbalance",

    # Past returns
    "return_500ms",
    "return_1s",
    "return_2s",
    "return_5s",
    "return_10s",

    # Recent volatility / activity
    "volatility_5s",
    "volatility_30s",
    "avg_events_5s",
    "avg_executions_5s",
]

print("Number of features:", len(feature_cols))

Number of features: 21


In [11]:
for asset in assets:
    train = splits[asset]["train"]

    print(f"\n{'='*50}")
    print(asset)
    print("="*50)

    # Missing values
    missing = train[feature_cols].isna().sum()

    print("\nFeatures with missing values:")
    print(missing[missing > 0])

    # Infinite values
    inf_counts = np.isinf(
        train[feature_cols].select_dtypes(include=np.number)
    ).sum()

    print("\nFeatures with infinite values:")
    print(inf_counts[inf_counts > 0])


TSLA

Features with missing values:
return_500ms           3
return_1s              6
return_2s             12
return_5s             30
return_10s            60
volatility_5s         30
volatility_30s       180
avg_events_5s         27
avg_executions_5s     27
dtype: int64

Features with infinite values:
Series([], dtype: int64)

AMZN

Features with missing values:
return_500ms           3
return_1s              6
return_2s             12
return_5s             30
return_10s            60
volatility_5s         30
volatility_30s       180
avg_events_5s         27
avg_executions_5s     27
dtype: int64

Features with infinite values:
Series([], dtype: int64)

BRK.B

Features with missing values:
return_500ms           3
return_1s              6
return_2s             12
return_5s             30
return_10s            60
volatility_5s         30
volatility_30s       180
avg_events_5s         27
avg_executions_5s     27
dtype: int64

Features with infinite values:
Series([], dtype: int64)


In [12]:
target_cols = {
    "TSLA": [
        "target_500ms",
        "target_2s",
        "target_10s",
    ],

    "AMZN": [
        "target_500ms",
        "target_2s",
        "target_10s",
    ],

    "BRK.B": [
        "target_5s",
        "target_10s",
        "target_30s",
    ],
}

for asset in assets:
    print(f"\n{asset}")

    for target in target_cols[asset]:
        train = splits[asset]["train"]

        print(
            target,
            "| missing:",
            train[target].isna().sum()
        )


TSLA
target_500ms | missing: 3
target_2s | missing: 12
target_10s | missing: 60

AMZN
target_500ms | missing: 3
target_2s | missing: 12
target_10s | missing: 60

BRK.B
target_5s | missing: 30
target_10s | missing: 60
target_30s | missing: 180


In [13]:
def make_xy(df, feature_cols, target_col):
    # Row is usable only if every feature AND the target exist
    valid = (
        df[feature_cols].notna().all(axis=1)
        & df[target_col].notna()
    )

    X = df.loc[valid, feature_cols].copy()
    y = df.loc[valid, target_col].copy()

    return X, y

In [14]:
for asset in assets:
    print(f"\n{'='*50}")
    print(asset)
    print("="*50)

    for target in target_cols[asset]:

        X_train, y_train = make_xy(
            splits[asset]["train"],
            feature_cols,
            target
        )

        X_val, y_val = make_xy(
            splits[asset]["val"],
            feature_cols,
            target
        )

        print(
            target,
            "| Train:", len(X_train),
            "| Validation:", len(X_val)
        )


TSLA
target_500ms | Train: 140217 | Validation: 46739
target_2s | Train: 140208 | Validation: 46736
target_10s | Train: 140160 | Validation: 46720

AMZN
target_500ms | Train: 140217 | Validation: 46739
target_2s | Train: 140208 | Validation: 46736
target_10s | Train: 140160 | Validation: 46720

BRK.B
target_5s | Train: 140190 | Validation: 46730
target_10s | Train: 140160 | Validation: 46720
target_30s | Train: 140040 | Validation: 46680


## Feature Redundancy Check

Pairwise feature correlations are examined using the training data only.
This provides a simple diagnostic for highly redundant predictors before
fixing the feature representation used by the downstream models.

In [15]:
for asset in assets:
    train = splits[asset]["train"]

    corr = (
        train[feature_cols]
        .corr()
        .abs()
    )

    pairs = []

    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            value = corr.iloc[i, j]

            if value >= 0.90:
                pairs.append({
                    "feature_1": feature_cols[i],
                    "feature_2": feature_cols[j],
                    "abs_correlation": value
                })

    high_corr = pd.DataFrame(pairs)

    if not high_corr.empty:
        high_corr = high_corr.sort_values(
            "abs_correlation",
            ascending=False
        )

    print(f"\n{asset} — highly correlated feature pairs")
    display(high_corr)


TSLA — highly correlated feature pairs


,feature_1,feature_2,abs_correlation
1,event_count,add_count,0.941024
2,event_count,delete_count,0.928505
0,imbalance_l1,microprice_deviation,0.914633



AMZN — highly correlated feature pairs


,feature_1,feature_2,abs_correlation
0,event_count,add_count,0.941904
1,event_count,delete_count,0.900815



BRK.B — highly correlated feature pairs


,feature_1,feature_2,abs_correlation
1,event_count,delete_count,0.995609
0,event_count,add_count,0.994046
2,add_count,delete_count,0.981541


## Final Modeling Feature Set

The initial candidate representation contains 21 interpretable predictors spanning
LOB state, message/order flow, recent price dynamics, volatility, and market activity.

The redundancy analysis above is performed using the training period only. It shows
that `add_count` and `delete_count` are strongly correlated with the overall
`event_count` across the assets, with absolute correlations above approximately 0.90
and particularly strong redundancy for BRK.B. To avoid retaining several variables
that largely measure the same short-term message intensity, `add_count` and
`delete_count` are excluded from the final modeling representation.

The resulting **19-feature set** retains four complementary sources of information:

- **LOB state (6):** spread, level-1 imbalance, level-5 imbalance, five-level bid
  depth, five-level ask depth, and microprice deviation. These characterize current
  liquidity, depth asymmetry, and near-book price pressure.

- **Order flow and activity (4):** event count, execution count, normalized book
  flow, and execution imbalance. These describe recent market activity and
  directional asymmetry in book updates and executions.

- **Recent price dynamics (5):** backward-looking log returns over 500 ms, 1 s,
  2 s, 5 s, and 10 s. Multiple look-back scales allow the models to capture
  short-lived continuation or reversal patterns.

- **Local market regime (4):** 5 s and 30 s volatility together with 5 s average
  event and execution activity. These provide context about recent volatility and
  trading intensity.

The feature set is therefore not the result of an automatic feature-selection
algorithm. It is a compact, economically interpretable representation designed to
cover complementary market-microstructure mechanisms while removing clear message-count
redundancy. All temporal predictors are backward-looking.

In [16]:
model_features = [
    # LOB state
    "spread",
    "imbalance_l1",
    "imbalance_l5",
    "bid_depth_5",
    "ask_depth_5",
    "microprice_deviation",

    # Order flow / activity
    "event_count",
    "execution_count",
    "normalized_book_flow",
    "execution_imbalance",

    # Past returns
    "return_500ms",
    "return_1s",
    "return_2s",
    "return_5s",
    "return_10s",

    # Volatility / recent activity
    "volatility_5s",
    "volatility_30s",
    "avg_events_5s",
    "avg_executions_5s",
]

print("Final baseline features:", len(model_features))

Final baseline features: 19


## Save Processed Datasets

The completed asset-day datasets are saved separately so that the chronological
structure remains explicit and the same processed inputs can be reused
consistently by the downstream notebooks.

In [17]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for (asset, date), df in daily_data.items():
    safe_asset = asset.replace(".", "_")

    path = OUTPUT_DIR / f"{safe_asset}_{date}.parquet"

    df.to_parquet(path)

print("Processed datasets saved to:", OUTPUT_DIR)

Processed datasets saved to: ../data/processed
